# 02 - ARIMA (SARIMAX) Model

> Seasonal ARIMA — the classical statistical baseline for time series forecasting.

**Model**: SARIMAX(1,1,1)(1,1,1,12) — captures both trend and yearly seasonality.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
COLORS = {'primary':'#003B73','alert':'#BF212F','success':'#27AE60'}
print("Libraries loaded")

## 1. Load Prepared Data

In [ ]:
train = pd.read_csv('../data/train_ts.csv', parse_dates=['ds'])
test = pd.read_csv('../data/test_ts.csv', parse_dates=['ds'])
train_idx = train.set_index('ds')
test_idx = test.set_index('ds')
print(f"Train: {len(train)} months | Test: {len(test)} months")

## 2. Fit SARIMAX Model

In [ ]:
model = SARIMAX(train_idx['y'], order=(1,1,1), seasonal_order=(1,1,1,12),
                enforce_stationarity=False, enforce_invertibility=False)
results = model.fit(disp=False)
print(results.summary().tables[1])

## 3. Predict on Test Set

In [ ]:
pred = results.predict(start=test_idx.index[0], end=test_idx.index[-1])
y_true = test_idx['y'].values
y_pred = pred.values

mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"ARIMA Test Metrics:")
print(f"  MAPE:  {mape:.2f}%")
print(f"  MAE:   ${mae:,.2f}")
print(f"  RMSE:  ${rmse:,.2f}")
print(f"  R2:    {r2:.4f}")

## 4. Visualize Predictions

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(train['ds'], train['y'], label='Train', color=COLORS['primary'], linewidth=2)
plt.plot(test['ds'], y_true, label='Actual (Test)', color='black', linewidth=2, marker='o')
plt.plot(test['ds'], y_pred, label=f'ARIMA Pred (MAPE={mape:.1f}%)', color=COLORS['alert'], linestyle='--', linewidth=2, marker='s')
plt.title('ARIMA (SARIMAX) Forecast vs Actuals', fontsize=15, fontweight='bold')
plt.xlabel('Date'); plt.ylabel('Monthly Sales ($)'); plt.legend()
plt.tight_layout()
plt.savefig('../visualizations/arima_forecast.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Save Model & Predictions

In [ ]:
# Save predictions
pred_df = pd.DataFrame({'ds': test['ds'].values, 'actual': y_true, 'predicted': y_pred})
pred_df.to_csv('../predictions/arima_predictions.csv', index=False)

# Save model
results.save('../models/arima_model.pkl')

print("Saved: predictions/arima_predictions.csv")
print("Saved: models/arima_model.pkl")
print("\nProceed to 03_Prophet_Model.ipynb")